# Hotel Booking Analysis — Practical Notebook

This notebook reproduces the practical steps for the project report you uploaded. It performs data loading, preprocessing, exploratory data analysis (EDA), and creates the same figures discussed in your report. **Place `hotel_bookings.csv` in the same folder as this notebook** or follow the Kaggle instructions below to download it.

If you want a single `.py` script instead, a companion `Hotel_Booking_Analysis.py` has been saved next to this notebook.

---


## How to get the dataset

1. Download manually from Kaggle: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand (choose `hotel_bookings.csv`) and put it beside this notebook.

2. Or use the Kaggle API (requires kaggle CLI and API token). Example commands:

```python
# pip install kaggle
# then:
# kaggle datasets download -d jessemostipak/hotel-booking-demand
# unzip hotel-booking-demand.zip
```


In [ ]:
# Standard imports and helper functions
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 120)

def load_dataset(path='hotel_bookings.csv'):
    if os.path.exists(path):
        print(f"Loading dataset from {path}")
        return pd.read_csv(path)
    else:
        raise FileNotFoundError(f"Dataset not found at {path} - place 'hotel_bookings.csv' beside this notebook.")

# small helper to save figures
def save_fig(fig, name):
    outdir = 'notebook_outputs'
    os.makedirs(outdir, exist_ok=True)
    p = os.path.join(outdir, name)
    fig.savefig(p, bbox_inches='tight')
    print('Saved:', p)


In [ ]:
# Load dataset
try:
    df = load_dataset('hotel_bookings.csv')
    print('Dataset shape:', df.shape)
    display(df.head(3))
except Exception as e:
    print(e)
    df = None


## Preprocessing & Feature Engineering
We perform typical cleaning, missing-value handling, and create features like total_guests and arrival_month_num.

In [ ]:
if df is not None:
    # Copy to avoid modifying original
    data = df.copy()

    # Basic types and missing values
    print('\nMissing values per column:')
    print(data.isnull().sum().sort_values(ascending=False).head(20))

    # Fill children NaN with 0 and convert types
    if 'children' in data.columns:
        data['children'] = data['children'].fillna(0).astype(int)
    # Fill agent/company with 0 and convert
    for col in ['agent','company']:
        if col in data.columns:
            data[col] = data[col].fillna(0).astype(int)

    # Create total_guests
    if set(['adults','children','babies']).issubset(data.columns):
        data['total_guests'] = data['adults'] + data['children'] + data['babies']

    # Convert arrival_date_month to categorical ordered month number
    if 'arrival_date_month' in data.columns:
        months = ['January','February','March','April','May','June','July','August','September','October','November','December']
        data['arrival_month_num'] = data['arrival_date_month'].apply(lambda x: months.index(x)+1 if x in months else np.nan)

    # Quick view
    display(data[['is_canceled','hotel','adr','arrival_date_month','arrival_month_num','lead_time','total_guests']].head())
else:
    print('Dataset not loaded; cannot preprocess.')


## EDA — Cancellation distribution and Hotel type comparison

In [ ]:
if df is not None:
    data = data  # already defined earlier
    fig = plt.figure(figsize=(6,4))
    ax = fig.add_subplot(1,1,1)
    canceled_counts = data['is_canceled'].value_counts().sort_index()
    ax.bar(canceled_counts.index.astype(str), canceled_counts.values)
    ax.set_xticks([0,1])
    ax.set_xticklabels(['Not Canceled','Canceled'])
    ax.set_ylabel('Number of bookings')
    ax.set_title('Reservation Status: Canceled vs Not Canceled')
    save_fig(fig, 'fig_reservation_status.png')
    plt.show()


In [ ]:
if df is not None:
    fig = plt.figure(figsize=(7,4))
    ax = fig.add_subplot(1,1,1)
    ctab = data.groupby(['hotel','is_canceled']).size().unstack(fill_value=0)
    ctab.plot(kind='bar', stacked=False, ax=ax)
    ax.set_ylabel('Bookings count')
    ax.set_title('Reservation Status: City Hotel vs Resort Hotel')
    save_fig(fig, 'fig_city_vs_resort.png')
    plt.show()


In [ ]:
if df is not None:
    fig = plt.figure(figsize=(8,4))
    ax = fig.add_subplot(1,1,1)
    grouped = data.groupby(['arrival_month_num','hotel'])['adr'].mean().unstack()
    grouped.plot(ax=ax)
    ax.set_xlabel('Month number')
    ax.set_ylabel('Average Daily Rate (ADR)')
    ax.set_title('Average Daily Rate: City Hotel vs Resort')
    save_fig(fig, 'fig_adr_by_hotel.png')
    plt.show()


In [ ]:
if df is not None:
    fig = plt.figure(figsize=(9,4))
    ax = fig.add_subplot(1,1,1)
    month_ct = data.groupby(['arrival_month_num','is_canceled']).size().unstack(fill_value=0)
    month_ct.plot(kind='bar', ax=ax)
    ax.set_xlabel('Month number')
    ax.set_ylabel('Bookings count')
    ax.set_title('Reservation Status per Month')
    save_fig(fig, 'fig_reservation_per_month.png')
    plt.show()


In [ ]:
if df is not None and 'country' in data.columns:
    top_countries = data[data['is_canceled']==1]['country'].value_counts().head(10)
    fig = plt.figure(figsize=(7,4))
    ax = fig.add_subplot(1,1,1)
    ax.barh(top_countries.index.astype(str)[::-1], top_countries.values[::-1])
    ax.set_xlabel('Canceled bookings')
    ax.set_title('Top 10 Countries with Reservation Cancellations')
    save_fig(fig, 'fig_top_countries_canceled.png')
    plt.show()


In [ ]:
if df is not None:
    fig = plt.figure(figsize=(6,4))
    ax = fig.add_subplot(1,1,1)
    means = data.groupby('is_canceled')['adr'].mean()
    ax.bar(['Not Canceled','Canceled'], means.values)
    ax.set_ylabel('Average Daily Rate (ADR)')
    ax.set_title('Average Daily Rate: Canceled vs Not Canceled')
    save_fig(fig, 'fig_adr_canceled_vs_not.png')
    plt.show()


In [ ]:
# Save a cleaned sample and also write a companion .csv for quick preview
if df is not None:
    sample = data.sample(n=min(200, len(data)), random_state=42)
    os.makedirs('notebook_outputs', exist_ok=True)
    sample.to_csv('notebook_outputs/sample_cleaned_preview.csv', index=False)
    print('Saved sample preview to notebook_outputs/sample_cleaned_preview.csv')


## Notes & How to present to your teacher

- Show the notebook cells in order while explaining each step: loading, cleaning, EDA, graphs, and insights.
- The `notebook_outputs` folder contains PNG figures and a sample CSV.
- To produce a video: run the notebook and record the screen while you explain each plot (or use `jupyter nbconvert --to slides` to export presentation slides).
